In [11]:
!pip install -q datasets huggingface_hub

In [12]:
# /kaggle/input/datasets/enddl22/deepnir-nir-rgb-sen12ms-dataset

In [13]:
#  Code for full folder preprocessing:--------

# import os
# import glob
# import cv2
# import numpy as np
# from tqdm import tqdm

# INPUT_DIR = "/kaggle/input/datasets/enddl22/deepnir-nir-rgb-sen12ms-dataset/sen12ms_SUMMER" 
# OUTPUT_DIR = "/kaggle/working/spectragan_dataset"

# # 1. Create the structured directories
# splits = ["train", "test", "val"]
# for split in splits:
#     os.makedirs(os.path.join(OUTPUT_DIR, split, "hr"), exist_ok=True)
#     os.makedirs(os.path.join(OUTPUT_DIR, split, "lr"), exist_ok=True)

# print(f"Directory structure created at {OUTPUT_DIR}")

# # 2. Gather files by split
# print("Scanning Kaggle directories for PNG files by split...")

# # Using wildcards to catch both train_A and train_B, test_A and test_B, etc.
# train_files = glob.glob(os.path.join(INPUT_DIR, "*train*/**/*.png"), recursive=True)
# test_files = glob.glob(os.path.join(INPUT_DIR, "*test*/**/*.png"), recursive=True)
# val_files = glob.glob(os.path.join(INPUT_DIR, "*val*/**/*.png"), recursive=True)

# # 3. Apply a limit for tonight's test run (e.g., 800 train, 100 test, 100 val)
# # Remove these slicing brackets [:] later when you want to process the full 450k dataset!
# # train_files = train_files
# # test_files = test_files
# # val_files = val_files

# # Combine them into a list of tuples: (split_name, file_path)
# files_to_process = [("train", f) for f in train_files] + \
#                    [("test", f) for f in test_files] + \
#                    [("val", f) for f in val_files]

# print(f"Processing {len(train_files)} Train, {len(test_files)} Test, and {len(val_files)} Val images.")

# valid_pairs = {"train": 0, "test": 0, "val": 0}

# # 4. The Processing Loop
# for split, file_path in tqdm(files_to_process, desc="Degrading Images"):
#     try:
#         hr_image_bgr = cv2.imread(file_path)
        
#         if hr_image_bgr is None:
#             continue
            
#         # CLEANING: Skip edge tiles
#         if np.max(hr_image_bgr) < 10:
#             continue
            
#         # DEGRADATION: Shrink from 256x256 to 64x64
#         height, width = hr_image_bgr.shape[:2]
#         lr_image_bgr = cv2.resize(hr_image_bgr, (width // 4, height // 4), interpolation=cv2.INTER_CUBIC)
        
#         # SAVE: Route to the correct split folder
#         filename = os.path.basename(file_path)
#         hr_out_path = os.path.join(OUTPUT_DIR, split, "hr", filename)
#         lr_out_path = os.path.join(OUTPUT_DIR, split, "lr", filename)
        
#         cv2.imwrite(hr_out_path, hr_image_bgr)
#         cv2.imwrite(lr_out_path, lr_image_bgr)
        
#         valid_pairs[split] += 1

#     except Exception as e:
#         pass

# print("\nSuccess! Dataset generated:")
# for split, count in valid_pairs.items():
#     print(f" - {split.capitalize()}: {count} pairs")

In [ ]:
# code for chunk of a folder preprocessing:--------

import os
import glob
import cv2
import numpy as np
from tqdm import tqdm

INPUT_DIR = "/kaggle/input/datasets/enddl22/deepnir-nir-rgb-sen12ms-dataset/sen12ms_All_seasons" 
OUTPUT_DIR = "/kaggle/working/spectragan_dataset"

# 1. Create the structured directories
splits = ["train", "test", "val"]
for split in splits:
    os.makedirs(os.path.join(OUTPUT_DIR, split, "hr"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, split, "lr"), exist_ok=True)

print(f"Directory structure created at {OUTPUT_DIR}")

# --- THE "PART B" SEARCH ---
print("Scanning Kaggle directories for PART B files...")

# Notice we explicitly target folders ending in _A
train_files = glob.glob(os.path.join(INPUT_DIR, "*train_B*/**/*.png"), recursive=True)
test_files = glob.glob(os.path.join(INPUT_DIR, "*test_B*/**/*.png"), recursive=True)
val_files = glob.glob(os.path.join(INPUT_DIR, "*val_B*/**/*.png"), recursive=True)
# train_files = train_files[:100] 
# test_files = test_files[:100] 
# val_files = val_files[:100] 

# Remove the limits! We want the full halves now.
# Combine them into a list of tuples
files_to_process = [("train", f) for f in train_files] + \
                   [("test", f) for f in test_files] + \
                   [("val", f) for f in val_files]

print(f"Processing {len(train_files)} Train, {len(test_files)} Test, and {len(val_files)} Val images.")
valid_pairs = {"train": 0, "test": 0, "val": 0}

# 4. The Processing Loop
for split, file_path in tqdm(files_to_process, desc="Degrading Images"):
    try:
        hr_image_bgr = cv2.imread(file_path)
        
        if hr_image_bgr is None:
            continue
            
        # CLEANING: Skip edge tiles
        if np.max(hr_image_bgr) < 10:
            continue
            
        # DEGRADATION: Shrink from 256x256 to 64x64
        height, width = hr_image_bgr.shape[:2]
        lr_image_bgr = cv2.resize(hr_image_bgr, (width // 4, height // 4), interpolation=cv2.INTER_CUBIC)
                
        # 1. Get the original filename
        original_filename = os.path.basename(file_path)
        
        # 2. Split the name and the extension (e.g., 'patch_01' and '.png')
        name, ext = os.path.splitext(original_filename)
        
        # 3. Create a unique filename using f-strings
        # This will look like: "patch_01_1.png", "patch_01_2.png", etc.
        unique_filename = f"{name}_{valid_pairs[split] + 1}{ext}"
        
        # 4. NOW use os.path.join with the new unique filename
        hr_out_path = os.path.join(OUTPUT_DIR, split, "hr", unique_filename)
        lr_out_path = os.path.join(OUTPUT_DIR, split, "lr", unique_filename)
        
        cv2.imwrite(hr_out_path, hr_image_bgr)
        cv2.imwrite(lr_out_path, lr_image_bgr)
        
        valid_pairs[split] += 1
    except Exception as e:
        pass

print("\nSuccess! Dataset generated:")
for split, count in valid_pairs.items():
    print(f" - {split.capitalize()}: {count} pairs")

Directory structure created at /kaggle/working/spectragan_dataset
Scanning Kaggle directories for PART B files...
Processing 144528 Train, 18067 Test, and 18067 Val images.


Degrading Images:  24%|██▍       | 43744/180662 [12:25<39:15, 58.12it/s]  

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
login(token = secret_value_0)

In [ ]:
import os
import glob
from datasets import Dataset, DatasetDict, Image
from kaggle_secrets import UserSecretsClient

OUTPUT_DIR = "/kaggle/working/spectragan_dataset"
splits = ["train", "test", "val"]

dataset_splits = {}

print("Building Hugging Face DatasetDict...")

for split in splits:
    print(f"Processing the '{split}' split...")
    hr_dir = os.path.join(OUTPUT_DIR, split, "hr")
    lr_dir = os.path.join(OUTPUT_DIR, split, "lr")
    
    # Sorting ensures LR and HR pairs match up perfectly
    hr_images = sorted(glob.glob(os.path.join(hr_dir, "*.png")))
    lr_images = sorted(glob.glob(os.path.join(lr_dir, "*.png")))
    
    if len(hr_images) == 0:
        print(f"Warning: No images found for {split}. Skipping.")
        continue
        
    assert len(hr_images) == len(lr_images), f"Mismatch in {split} counts!"
    
    # NEW STEP: Extract the season from the filename for every image in this split
    # Since the name is like "ROIs1158_spring_1.png", splitting by '_' gets "spring" at index 1
    seasons = [os.path.basename(path).split('_')[1] for path in hr_images]
    
    # Add the seasons list to our data dictionary
    data_dict = {
        "lr": lr_images,
        "hr": hr_images,
        "season": seasons  # <-- New column!
    }
    
    # Create the dataset
    split_dataset = Dataset.from_dict(data_dict)
    
    # Cast the image columns (Hugging Face handles the string 'season' column automatically)
    split_dataset = split_dataset.cast_column("lr", Image())
    split_dataset = split_dataset.cast_column("hr", Image())
    
    dataset_splits[split] = split_dataset

hf_dataset_dict = DatasetDict(dataset_splits)

print("\nDataset Structure:")
print(hf_dataset_dict)

# --- UPLOAD TO THE HUB ---
REPO_NAME = "ParamDev/SpectraGAN-SEN12MS_AllSeasons_partB"

print(f"\nPushing dataset to {REPO_NAME}...")
hf_dataset_dict.push_to_hub(REPO_NAME)

print("\nUpload complete! You can now load this anywhere using:")
print(f"dataset = load_dataset('{REPO_NAME}')")